## Stepwise Linear Regression – Sunoco Gasoline Demand (revised)
Builds two multiple linear regression equations for weekly `Demand`: one using **forward selection** and one using **backward selection**, then uses both to forecast the 2025–2026 weeks.

**What changed from the previous version (to stop negative Demand values):**
1. **No blank inputs in the forecast weeks.** If a predictor is blank for a 2025–2026 week, it effectively gets plugged in as 0. With a constant of about −48,000 and `us_population_thous` worth about +60,000 in the old equation, one blank cell pushes the forecast deeply negative. Gaps in the forecast weeks are now filled with the last known value.
2. **Near-duplicate predictors are removed before stepwise selection** (Variance Inflation Factor check). WTI vs. Brent, GDP vs. PCE, and `Week #` vs. population move together almost perfectly. The old model let them cancel each other out with big opposite-sign coefficients, which is what produced the huge negative constant and made forecasts swing wildly outside the training period.
3. **Forecasts are computed here in Python and checked**, then saved to Excel, so nothing has to be typed into a spreadsheet by hand.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', None)

### Load the data and split it
Training weeks = weeks with a known `Demand` (1991–2024). Forecast weeks = the 2025–2026 rows where `Demand` is blank on purpose.

In [ ]:
df = pd.read_excel("Sunoco Training_complete_NEW.xlsx", sheet_name="Target")

# columns that are not predictors (date, empty columns, earlier model outputs)
drop_cols = ["Date", "Model C", "Simple Linear Regression Model"]
not_predictors = [c for c in drop_cols if c in df.columns] + \
                 [c for c in df.columns if str(c).startswith("Unnamed")]
candidate_cols = [c for c in df.columns if c not in not_predictors + ["Demand"]]

train_mask = df["Demand"].notna()
forecast_mask = df["Demand"].isna()
if "Date" in df.columns:                       # ignore fully empty rows at the bottom of the sheet
    forecast_mask &= df["Date"].notna()

print("Training weeks:", train_mask.sum(), "| Forecast weeks:", forecast_mask.sum())

### Handle missing values
**Training weeks:** same rule as before – keep predictors that are available for at least 90% of the training weeks, then drop the few rows that still have gaps.

**Forecast weeks:** any blank predictor value is filled with its most recent known value (carried forward). This is the step that prevents blanks from turning into zeros and producing negative forecasts. The table printed below shows how many forecast weeks needed filling, so you can see which inputs are stale.

In [ ]:
coverage = df.loc[train_mask, candidate_cols].notna().mean()
print("Dropped (under 90% coverage in training weeks):")
print(coverage[coverage < 0.90].sort_values().round(2))

keep_cols = coverage[coverage >= 0.90].index.tolist()

train = df.loc[train_mask, keep_cols + ["Demand"]].dropna()
print("\nRows used for modeling:", len(train))

# forecast weeks: carry the last known value forward into any blanks
filled_all = df[keep_cols].ffill()
forecast_X = filled_all.loc[forecast_mask]
n_filled = df.loc[forecast_mask, keep_cols].isna().sum()
print("\nForecast weeks that needed a carried-forward value:")
print(n_filled[n_filled > 0] if (n_filled > 0).any() else "none – every forecast input was already filled in")

In [ ]:
outcome = train["Demand"]
predictors = train.drop(columns=["Demand"])
predictors.columns.tolist()

### Remove near-duplicate predictors (VIF check)
The Variance Inflation Factor (VIF) measures how well a predictor can be explained by the other predictors. A VIF above 10 means it is mostly a copy of the others. We repeatedly drop the predictor with the highest VIF until all are under 10.

Keeping duplicates doesn't improve the fit much, but it lets the model assign huge offsetting coefficients (e.g. +15.6 for WTI and −21.8 for Brent), and those blow up as soon as the inputs move outside the training range.

In [ ]:
def prune_vif(X, threshold=10.0, verbose=True):
    cols = list(X.columns)
    while len(cols) > 1:
        Xc = sm.add_constant(X[cols]).values
        vif = pd.Series([variance_inflation_factor(Xc, i) for i in range(1, Xc.shape[1])], index=cols)
        if vif.max() > threshold:
            worst = vif.idxmax()
            cols.remove(worst)
            if verbose:
                print(f"Drop {worst:32} VIF {vif.max():,.1f}")
        else:
            break
    return cols

vif_features = prune_vif(predictors, threshold=10.0)
predictors = predictors[vif_features]
print("\nPredictors passed to stepwise selection:", vif_features)

### Forward selection
Start with no predictors. Each round, try adding every remaining predictor and keep the one with the smallest p-value, as long as it is below `threshold_in`. Stop when nothing new qualifies.

In [ ]:
def forward_regression(X, y, threshold_in=0.05, verbose=True):
    included = []
    while True:
        excluded = [c for c in X.columns if c not in included]
        if not excluded:
            break
        pvals = pd.Series(index=excluded, dtype=float)
        for col in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [col]])).fit()
            pvals[col] = model.pvalues[col]
        if pvals.min() < threshold_in:
            best = pvals.idxmin()
            included.append(best)
            if verbose:
                print(f"Add  {best:32} p-value {pvals.min():.6f}")
        else:
            break
    return included

### Backward selection
Start with every predictor. Each round, drop the predictor with the largest p-value, as long as it is above `threshold_out`. Stop when every remaining predictor is significant.

In [ ]:
def backward_regression(X, y, threshold_out=0.05, verbose=True):
    included = list(X.columns)
    while included:
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvals = model.pvalues.drop("const")
        if pvals.max() > threshold_out:
            worst = pvals.idxmax()
            included.remove(worst)
            if verbose:
                print(f"Drop {worst:32} p-value {pvals.max():.6f}")
        else:
            break
    return included

### Helper: write the fitted model as an equation

In [ ]:
def print_equation(model, name="Demand"):
    eq = f"{name} = {model.params['const']:.4f}"
    for var, coef in model.params.drop("const").items():
        sign = "+" if coef >= 0 else "-"
        eq += f"\n        {sign} {abs(coef):.6g} * {var}"
    print(eq)

## Model 1: Forward selection

In [ ]:
forward_features = forward_regression(predictors, outcome, threshold_in=0.05)
print("\nSelected:", forward_features)

In [ ]:
forward_model = sm.OLS(outcome, sm.add_constant(predictors[forward_features])).fit()
forward_model.summary()

In [ ]:
print_equation(forward_model)

## Model 2: Backward selection

In [ ]:
backward_features = backward_regression(predictors, outcome, threshold_out=0.05)
print("\nSelected:", backward_features)

In [ ]:
backward_model = sm.OLS(outcome, sm.add_constant(predictors[backward_features])).fit()
backward_model.summary()

In [ ]:
print_equation(backward_model)

## Compare the two models

In [ ]:
comparison = pd.DataFrame({
    "Forward":  [len(forward_features),  forward_model.rsquared,  forward_model.rsquared_adj,  forward_model.aic,  np.sqrt(forward_model.mse_resid)],
    "Backward": [len(backward_features), backward_model.rsquared, backward_model.rsquared_adj, backward_model.aic, np.sqrt(backward_model.mse_resid)],
}, index=["# predictors", "R-squared", "Adj. R-squared", "AIC", "Std. error"])
comparison.round(4)

### Residuals and fitted values
Residuals (actual − fitted) are **supposed** to be a mix of positive and negative numbers; a negative residual just means that week's actual demand came in below the model's estimate. The thing that should never be negative is the fitted/forecast Demand itself, so that is what we check.

In [ ]:
print("Lowest fitted Demand (forward): ", round(forward_model.fittedvalues.min(), 1))
print("Lowest fitted Demand (backward):", round(backward_model.fittedvalues.min(), 1))
forward_model.resid.head(), forward_model.fittedvalues.head()

## Forecast the 2025–2026 weeks
Both equations are applied to the forecast weeks using the gap-filled inputs, then checked for negative values and saved to `Sunoco_forecasts.xlsx`.

In [ ]:
def predict(model, features):
    return model.predict(sm.add_constant(forecast_X[features], has_constant="add"))

forecasts = pd.DataFrame(index=forecast_X.index)
if "Date" in df.columns:
    forecasts["Date"] = df.loc[forecast_mask, "Date"]
forecasts["Forward model"]  = predict(forward_model,  forward_features)
forecasts["Backward model"] = predict(backward_model, backward_features)

for col in ["Forward model", "Backward model"]:
    n_neg = (forecasts[col] < 0).sum()
    print(f"{col:15} min {forecasts[col].min():10,.1f}   max {forecasts[col].max():10,.1f}   negative weeks: {n_neg}")

forecasts.to_excel("Sunoco_forecasts.xlsx", index=False)
forecasts.head(10)